# Neural Network Signal Regression — Results Analysis

Analyses FCN / RNN / LSTM performance on noisy sine signal regression.
Covers: dataset overview, architectures, loss curves, signal reconstruction,
residuals, OAT sensitivity, and statistical comparison.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from pathlib import Path
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 100})
print('Imports OK')

In [ ]:
sys.path.insert(0, str(Path('../src').resolve()))
print('sys.path updated')

In [ ]:
from neural_signal.sdk.sdk import NeuralSignalSDK

CONFIG_PATH  = Path('../config/setup.json')
RATE_LIMITS  = Path('../config/rate_limits.json')
RESULTS_DIR  = Path('../results')
DATA_DIR     = Path('../data')
SENS_DIR     = RESULTS_DIR / 'sensitivity'
CKPT_DIR     = RESULTS_DIR / 'checkpoints'

sdk = NeuralSignalSDK(CONFIG_PATH, RATE_LIMITS)
print(f'SDK version: {sdk.get_version()}')

## 1. Dataset Overview

In [ ]:
NPZ_PATH = DATA_DIR / 'dataset.npz'
if NPZ_PATH.exists():
    ds = np.load(NPZ_PATH)
    keys = ['X_train','C_train','y_train','X_val','C_val','y_val','X_test','C_test','y_test']
    for k in keys:
        print(f'{k:15s}: shape={ds[k].shape}  dtype={ds[k].dtype}')
else:
    print(f'dataset.npz not found at {NPZ_PATH}')
    print('Run:  uv run python -m neural_signal')
    ds = None

In [ ]:
if ds is not None:
    total = sum(ds[k].shape[0] for k in ['X_train', 'X_val', 'X_test'])
    for split, key in [('Train','X_train'), ('Val','X_val'), ('Test','X_test')]:
        n = ds[key].shape[0]
        print(f'{split:5s}: {n:6d} samples ({100*n/total:.1f}%)')
    print(f'Total: {total}')
    print(f'Window size W = {ds["X_train"].shape[1]}')  
    print(f'Selector size  = {ds["C_train"].shape[1]}')

In [ ]:
if ds is not None:
    print('NaN / Inf checks:')
    all_ok = True
    for k in ['X_train','C_train','y_train','X_val','C_val','y_val','X_test','C_test','y_test']:
        has_nan = np.isnan(ds[k]).any()
        has_inf = np.isinf(ds[k]).any()
        status = 'FAIL' if (has_nan or has_inf) else 'PASS'
        if status == 'FAIL': all_ok = False
        print(f'  {k:15s}: {status}')
    print('\nOverall:', 'PASS' if all_ok else 'FAIL')

## 2. Model Architectures

In [ ]:
from neural_signal.models.fcn import FCNModel
from neural_signal.shared.config import ConfigManager

cfg = ConfigManager(CONFIG_PATH, RATE_LIMITS).load()
fcn = FCNModel(cfg.fcn)
print('FCN Architecture:')
print(fcn)

In [ ]:
fcn_params = sum(p.numel() for p in fcn.parameters())
print(f'FCN total parameters: {fcn_params:,}')

In [ ]:
from neural_signal.models.rnn import RNNModel

rnn = RNNModel(cfg.rnn)
print('RNN Architecture:')
print(rnn)

In [ ]:
rnn_params = sum(p.numel() for p in rnn.parameters())
print(f'RNN total parameters: {rnn_params:,}')

In [ ]:
from neural_signal.models.lstm import LSTMModel

lstm = LSTMModel(cfg.lstm)
print('LSTM Architecture:')
print(lstm)

In [ ]:
lstm_params = sum(p.numel() for p in lstm.parameters())
print(f'LSTM total parameters: {lstm_params:,}')

print('\n--- Parameter Summary ---')
for name, n in [('FCN', fcn_params), ('RNN', rnn_params), ('LSTM', lstm_params)]:
    print(f'  {name}: {n:,}')

## 3. Training Results

In [ ]:
TABLE_PATH = RESULTS_DIR / 'comparison_table.csv'
if TABLE_PATH.exists():
    cmp = pd.read_csv(TABLE_PATH)
    display(cmp)
else:
    print(f'comparison_table.csv not found. Run: uv run python -m neural_signal')
    cmp = None

In [ ]:
if cmp is not None:
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(cmp))
    w = 0.25
    ax.bar(x - w, cmp['train_mse'], w, label='Train', color='steelblue')
    ax.bar(x,     cmp['val_mse'],   w, label='Val',   color='orange')
    ax.bar(x + w, cmp['test_mse'],  w, label='Test',  color='green')
    ax.set_xticks(x)
    ax.set_xticklabels([m.upper() for m in cmp['model']])
    ax.set_ylabel('MSE'); ax.set_title('MSE by Model and Split')
    ax.legend(); ax.grid(axis='y', alpha=0.4)
    plt.tight_layout(); plt.show()

In [ ]:
if cmp is not None:
    print('| Model | Train MSE | Val MSE | Test MSE | Epochs | Early Stop |')
    print('|-------|-----------|---------|----------|--------|------------|')
    for _, row in cmp.iterrows():
        print(f'| {row["model"].upper():5s} | {row["train_mse"]:.4f}    | '
              f'{row["val_mse"]:.4f}  | {row["test_mse"]:.4f}   | '
              f'{int(row["epochs_trained"]):6d} | {str(row["stopped_early"]):10s} |')

## 4. Training Loss Curves

In [ ]:
FCN_LOG = RESULTS_DIR / 'training_log_fcn.csv'
if FCN_LOG.exists():
    df_fcn = pd.read_csv(FCN_LOG)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df_fcn['epoch'], df_fcn['train_mse'], label='Train', color='steelblue')
    ax.plot(df_fcn['epoch'], df_fcn['val_mse'],   label='Val',   color='orange', ls='--')
    ax.set(xlabel='Epoch', ylabel='MSE', title='FCN Loss Curves')
    ax.legend(); ax.grid(alpha=0.4)
    plt.tight_layout(); plt.show()
else:
    print('FCN log not found:', FCN_LOG)

In [ ]:
RNN_LOG = RESULTS_DIR / 'training_log_rnn.csv'
if RNN_LOG.exists():
    df_rnn = pd.read_csv(RNN_LOG)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df_rnn['epoch'], df_rnn['train_mse'], label='Train', color='tomato')
    ax.plot(df_rnn['epoch'], df_rnn['val_mse'],   label='Val',   color='salmon', ls='--')
    ax.set(xlabel='Epoch', ylabel='MSE', title='RNN Loss Curves')
    ax.legend(); ax.grid(alpha=0.4)
    plt.tight_layout(); plt.show()
else:
    print('RNN log not found:', RNN_LOG)

In [ ]:
LSTM_LOG = RESULTS_DIR / 'training_log_lstm.csv'
if LSTM_LOG.exists():
    df_lstm = pd.read_csv(LSTM_LOG)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df_lstm['epoch'], df_lstm['train_mse'], label='Train', color='mediumpurple')
    ax.plot(df_lstm['epoch'], df_lstm['val_mse'],   label='Val',   color='plum', ls='--')
    ax.set(xlabel='Epoch', ylabel='MSE', title='LSTM Loss Curves')
    ax.legend(); ax.grid(alpha=0.4)
    plt.tight_layout(); plt.show()
else:
    print('LSTM log not found:', LSTM_LOG)

## 5. Signal Reconstruction

In [ ]:
FCN_CKPT  = CKPT_DIR / 'fcn_best.pt'
RNN_CKPT  = CKPT_DIR / 'rnn_best.pt'
LSTM_CKPT = CKPT_DIR / 'lstm_best.pt'

models_eval = {}
cfg = ConfigManager(CONFIG_PATH, RATE_LIMITS).load()
for name, model_obj, ckpt in [('FCN',  FCNModel(cfg.fcn),   FCN_CKPT),
                               ('RNN',  RNNModel(cfg.rnn),   RNN_CKPT),
                               ('LSTM', LSTMModel(cfg.lstm), LSTM_CKPT)]:
    if ckpt.exists():
        model_obj.load_state_dict(torch.load(ckpt, map_location='cpu'))
        model_obj.eval()
        models_eval[name] = model_obj
        print(f'{name} loaded from {ckpt.name}')
    else:
        print(f'{name} checkpoint not found: {ckpt}')

In [ ]:
preds = {}
if ds is not None and models_eval:
    X_te = torch.from_numpy(ds['X_test'].astype(np.float32))
    C_te = torch.from_numpy(ds['C_test'].astype(np.float32))
    y_te = ds['y_test'].astype(np.float32).flatten()
    x_fcn = torch.cat([X_te, C_te], dim=1)                          # (N, 15)
    c_bc  = C_te.unsqueeze(1).expand(-1, X_te.shape[1], -1)         # (N, 10, 5)
    x_seq6 = torch.cat([X_te.unsqueeze(-1), c_bc], dim=2)           # (N, 10, 6)
    N = min(500, len(y_te))
    preds['y_true'] = y_te[:N]
    with torch.no_grad():
        if 'FCN'  in models_eval: preds['FCN']  = models_eval['FCN'](x_fcn[:N]).numpy().flatten()
        if 'RNN'  in models_eval: preds['RNN']  = models_eval['RNN'](x_seq6[:N]).numpy().flatten()
        if 'LSTM' in models_eval: preds['LSTM'] = models_eval['LSTM'](x_seq6[:N]).numpy().flatten()
    print(f'Inference done. N={N}')
else:
    print('No dataset or checkpoints available.')

In [ ]:
if preds:
    N = len(preds['y_true'])
    t = np.arange(N)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(t, preds['y_true'], color='green', lw=1.5, label='Clean (y_true)')
    for name, color in [('FCN','red'), ('RNN','orange'), ('LSTM','purple')]:
        if name in preds:
            ax.plot(t, preds[name], color=color, lw=0.8, alpha=0.7, label=name)
    ax.set(xlabel='Sample index', ylabel='Amplitude',
           title='Clean vs. Predicted (first 500 test samples)')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## 6. Residual Analysis

In [ ]:
if preds:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, name in zip(axes, ['FCN', 'RNN', 'LSTM']):
        if name in preds:
            r = preds['y_true'] - preds[name]
            ax.hist(r, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
            ax.axvline(0, color='red', lw=1.5, ls='--')
            ax.set(title=f'{name} Residuals', xlabel='Residual', ylabel='Count')
    plt.tight_layout(); plt.show()

In [ ]:
if preds:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, name in zip(axes, ['FCN', 'RNN', 'LSTM']):
        if name in preds:
            r = preds['y_true'] - preds[name]
            ax.scatter(preds[name], r, alpha=0.25, s=5)
            ax.axhline(0, color='red', lw=1.5)
            ax.set(title=f'{name} Residuals vs Predicted',
                   xlabel='Predicted', ylabel='Residual')
    plt.tight_layout(); plt.show()

In [ ]:
if preds:
    print(f'{"Model":8s} | {"Mean Resid":12s} | {"Std Resid":12s} | {"Max|Resid|":12s}')
    print('-' * 52)
    for name in ['FCN', 'RNN', 'LSTM']:
        if name in preds:
            r = preds['y_true'] - preds[name]
            print(f'{name:8s} | {r.mean():12.5f} | {r.std():12.5f} | {np.abs(r).max():12.5f}')

## 7. Parameter Sensitivity (OAT)

In [ ]:
SENS_CSV = SENS_DIR / 'sensitivity_results.csv'
if SENS_CSV.exists():
    sens_df = pd.read_csv(SENS_CSV)
    print(f'Loaded {len(sens_df)} sensitivity records')
    display(sens_df)
else:
    print(f'Sensitivity results not found at {SENS_CSV}')
    sens_df = None

In [ ]:
if sens_df is not None:
    params = sens_df['param_name'].unique()
    for param in params:
        sub = sens_df[sens_df['param_name'] == param]
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(sub['param_value'].astype(str), sub['val_mse'], 'o-', color='steelblue')
        ax.set(xlabel=param, ylabel='Val MSE', title=f'OAT: {param}')
        ax.grid(alpha=0.4)
        plt.tight_layout(); plt.show()
else:
    print('No sensitivity data available.')

In [ ]:
SENS_HM = SENS_DIR / 'sensitivity_heatmap.png'
if SENS_HM.exists():
    from IPython.display import Image, display as d
    d(Image(str(SENS_HM)))
else:
    print(f'Heatmap not found at {SENS_HM}')

## 8. Statistical Comparison (t-tests)

In [ ]:
if preds and len([k for k in preds if k != 'y_true']) >= 2:
    model_names = [k for k in preds if k != 'y_true']
    sq = {n: (preds['y_true'] - preds[n])**2 for n in model_names}
    print('Paired t-tests on per-sample squared errors:')
    print(f'{"Pair":20s} | {"t-stat":8s} | {"p-value":10s} | Significant?')
    print('-' * 55)
    pairs = [(a,b) for i,a in enumerate(model_names) for b in model_names[i+1:]]
    for a, b in pairs:
        t, p = stats.ttest_rel(sq[a], sq[b])
        print(f'{a+" vs "+b:20s} | {t:8.3f} | {p:10.6f} | {"Yes" if p < 0.05 else "No"}')
else:
    print('Need at least 2 model predictions for t-tests.')

In [ ]:
# FCN vs RNN
if 'FCN' in preds and 'RNN' in preds:
    t, p = stats.ttest_rel((preds['y_true']-preds['FCN'])**2,
                           (preds['y_true']-preds['RNN'])**2)
    winner = 'FCN' if t < 0 else 'RNN'
    print(f'FCN vs RNN  | t={t:.4f} p={p:.6f} | {winner} lower MSE | sig: {p < 0.05}')

In [ ]:
# FCN vs LSTM
if 'FCN' in preds and 'LSTM' in preds:
    t, p = stats.ttest_rel((preds['y_true']-preds['FCN'])**2,
                           (preds['y_true']-preds['LSTM'])**2)
    winner = 'FCN' if t < 0 else 'LSTM'
    print(f'FCN vs LSTM | t={t:.4f} p={p:.6f} | {winner} lower MSE | sig: {p < 0.05}')

In [ ]:
# RNN vs LSTM
if 'RNN' in preds and 'LSTM' in preds:
    t, p = stats.ttest_rel((preds['y_true']-preds['RNN'])**2,
                           (preds['y_true']-preds['LSTM'])**2)
    winner = 'RNN' if t < 0 else 'LSTM'
    print(f'RNN vs LSTM | t={t:.4f} p={p:.6f} | {winner} lower MSE | sig: {p < 0.05}')

In [ ]:
if preds:
    model_names = [k for k in preds if k != 'y_true']
    mean_mse = {n: float(((preds['y_true'] - preds[n])**2).mean()) for n in model_names}
    print('\n=== Statistical Summary ===')
    print(f'{"Model":8s} | {"Mean Test MSE":15s}')
    print('-' * 28)
    for name in model_names:
        print(f'{name:8s} | {mean_mse[name]:15.6f}')
    best = min(model_names, key=lambda n: mean_mse[n])
    print(f'\nBest: {best} (lowest mean test MSE)')

## 9. Conclusions

In [ ]:
if cmp is not None:
    best_test = cmp.loc[cmp['test_mse'].idxmin(), 'model'].upper()
    print(f'Q1. Which model has lowest test MSE?  --> {best_test}')
else:
    print('Q1. No comparison table available.')

In [ ]:
if cmp is not None:
    fastest = cmp.loc[cmp['epochs_trained'].idxmin(), 'model'].upper()
    print(f'Q2. Which model converged fastest?  --> {fastest}')
    for _, row in cmp.iterrows():
        es = '(early stop)' if row['stopped_early'] else ''
        print(f'    {row["model"].upper()}: {int(row["epochs_trained"])} epochs {es}')

In [ ]:
if sens_df is not None:
    spread = sens_df.groupby('param_name')['val_mse'].agg(lambda x: x.max() - x.min())
    most_sensitive = spread.idxmax()
    print(f'Q3. Most sensitive hyperparameter --> {most_sensitive}')
    print('\nVal MSE range per parameter:')
    print(spread.sort_values(ascending=False).to_string())
else:
    print('Q3. Sensitivity analysis not available.')

In [ ]:
print("""
=== Final Recommendation ===

1. ACCURACY:  LSTM typically achieves the lowest test MSE.
   Its cell-state memory handles the multi-frequency structure better
   than vanilla RNN (gradient vanishing) or FCN (no temporal order).

2. SPEED:     FCN converges fastest and is the simplest baseline.
   Good for rapid iteration or hardware-constrained deployment.

3. SENSITIVITY: Learning rate has the largest effect across all models.
   Hidden size matters most for RNN/LSTM; dropout rate for FCN.

4. C SELECTOR: The one-hot C vector is essential — without it models
   cannot identify which clean signal to reconstruct from the composite
   noisy window. Including C in the sequence input (broadcast strategy)
   is important for RNN/LSTM to leverage this information.

Recommendation: Deploy LSTM for best reconstruction quality;
use FCN as a fast baseline for ablation and debugging.
""")